# ESG Framework Mapping Portfolio
### SASB Standards: Industry x Sustainability Topic Analysis
---
**Framework:** SASB (Sustainability Accounting Standards Board)  
**Coverage:** 11 Sectors, 77 Industries, 27 ESG Topics, 5 Dimensions  
**Tools:** Python, Pandas, Plotly  
**Source:** Scraped from SASB public documentation


In [ ]:
import csv
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = r'C:/Users/Hp/ESG_Portfolio/sasb_esg_mapping.csv'
with open(DATA_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    df = []
    for row in reader:
        row['Materiality_Score'] = int(row['Materiality_Score'])
        row['Is_Material'] = str(row['Is_Material']).strip().lower() in ('1', 'true', 'yes')
        df.append(row)

print('Rows:', len(df))
print('Sectors:', len({r['Sector'] for r in df}))
print('Industries:', len({r['Industry'] for r in df}))
print('ESG Topics:', len({r['ESG_Topic'] for r in df}))
df[:5]

---
## Project 1 - ESG Materiality Heatmap
**Which sectors have the highest ESG materiality across each dimension?**

In [ ]:
from collections import defaultdict

pivot = defaultdict(lambda: defaultdict(list))
for row in df:
    pivot[row['Sector']][row['ESG_Dimension']].append(row['Materiality_Score'])

sectors = sorted(pivot)
dimensions = sorted({d for dims in pivot.values() for d in dims})
heatmap_data = [[sum(pivot[s][d]) / len(pivot[s][d]) for d in dimensions] for s in sectors]

fig = px.imshow(
    heatmap_data,
    x=dimensions, y=sectors,
    color_continuous_scale='RdYlGn',
    title='ESG Materiality Heatmap - Sector vs Dimension (Score 1-5)',
    labels=dict(color='Score'),
    aspect='auto', text_auto='.1f'
)
fig.update_layout(template='plotly_dark', paper_bgcolor='#0b1a14', height=500)

fig.show()print('Key Insight: Extractives leads on Environment; Financials leads on Governance.')

---
## Project 2 - Top ESG Topics by Industry Count
**Which ESG topics are most material across ALL industries?**

In [ ]:
color_map = {
    'Environment': '#00c896',
    'Social Capital': '#69c0ff',
    'Human Capital': '#ffc864',
    'Business Model & Innovation': '#ff8c69',
    'Leadership & Governance': '#b39ddb'
}

topic_counts_map = {}
for row in df:
    if row['Is_Material']:
        key = (row['ESG_Topic'], row['ESG_Dimension'])
        topic_counts_map.setdefault(key, set()).add(row['Industry'])

topic_counts = [
    {'ESG_Topic': topic, 'ESG_Dimension': dimension, 'Industry_Count': len(industries)}
    for (topic, dimension), industries in topic_counts_map.items()
]

topic_counts.sort(key=lambda x: x['Industry_Count'], reverse=True)

fig = px.bar(
    topic_counts,
    x='Industry_Count', y='ESG_Topic', color='ESG_Dimension',
    color_discrete_map=color_map, orientation='h',
    title='Most Material ESG Topics - Number of Industries Affected',
    labels={'Industry_Count':'No. of Industries','ESG_Topic':'ESG Topic'}
)
fig.update_layout(

    template='plotly_dark',print('Key Insight: Business Ethics and Data Security material for nearly every industry.')

    paper_bgcolor='#0b1a14',fig.show()

    plot_bgcolor='#112a1e',)

    height=600,    yaxis=dict(autorange='reversed')

---
## Project 3 - Industry Deep-Dive ESG Profile
**Full ESG materiality profile for a specific industry**

In [ ]:
INDUSTRY = 'Oil & Gas - Exploration & Production'

ind_df = sorted(
    [row for row in df if row['Industry'] == INDUSTRY],
    key=lambda row: row['Materiality_Score'],
    reverse=True
)

fig = px.bar(
    ind_df, x='Materiality_Score', y='ESG_Topic', color='ESG_Dimension',
    color_discrete_map=color_map, orientation='h',
    title='ESG Profile: ' + INDUSTRY,
    labels={'Materiality_Score':'Materiality Score (1-5)','ESG_Topic':''}
)
fig.add_vline(x=3, line_dash='dash', line_color='white', line_width=1,
    annotation_text='Material threshold')
fig.update_layout(
    template='plotly_dark', paper_bgcolor='#0b1a14',
    plot_bgcolor='#112a1e', height=600, yaxis=dict(autorange='reversed')
)
fig.show()


material = [row['ESG_Topic'] for row in ind_df if row['Is_Material']]    print(' -', t)

print('Material topics (' + str(len(material)) + '):')for t in material:

---
## Project 4 - Sector ESG Radar Chart
**How do sectors compare across all 5 ESG dimensions?**

In [ ]:
COMPARE = [
    'Extractives & Minerals Processing',
    'Technology & Communications',
    'Financials',
    'Food & Beverage',
    'Transportation'
]
dimensions = sorted({row['ESG_Dimension'] for row in df})
colors = ['#ff6b6b','#69c0ff','#ffc864','#00c896','#b39ddb']

fig = go.Figure()
for sector, color in zip(COMPARE, colors):
    scores = []
    for d in dimensions:
        values = [row['Materiality_Score'] for row in df if row['Sector'] == sector and row['ESG_Dimension'] == d]
        avg = round(sum(values) / len(values), 2) if values else 0
        scores.append(avg)
    scores.append(scores[0])
    fig.add_trace(go.Scatterpolar(
        r=scores, theta=dimensions+[dimensions[0]],
        fill='toself', name=sector, line=dict(color=color, width=2)))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,5])),
    title='ESG Dimension Radar - Sector Comparison',
    template='plotly_dark', paper_bgcolor='#0b1a14', height=550)
fig.show()
print('Key Insight: Extractives high on Environment; Tech leads on Social Capital.')

---
## Project 5 - ESG Materiality Rate by Sector
**Which sectors have the highest % of material ESG topics?**

In [ ]:
sector_totals = {}
for row in df:
    sector_totals.setdefault(row['Sector'], {'Total': 0, 'Material': 0})
    sector_totals[row['Sector']]['Total'] += 1
    sector_totals[row['Sector']]['Material'] += int(row['Is_Material'])

mat_rate = [
    {
        'Sector': sector,
        'Total': values['Total'],
        'Material': values['Material'],
        'Rate': round(values['Material'] / values['Total'] * 100, 1)
    }
    for sector, values in sector_totals.items()
]
mat_rate.sort(key=lambda row: row['Rate'], reverse=True)

fig = px.bar(
    mat_rate,
    x='Rate', y='Sector', orientation='h',
    color='Rate', color_continuous_scale='RdYlGn',
    title='ESG Materiality Rate by Sector (%)',
    labels={'Rate':'Materiality Rate (%)','Sector':''}
)
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0b1a14',
    plot_bgcolor='#112a1e',
    height=460,

    coloraxis_showscale=False,    print(f"{row['Sector']:<26} {row['Material']:>8} {row['Total']:>6} {row['Rate']:>6}")

    yaxis=dict(autorange='reversed')for row in mat_rate:

)print('Sector                     Material   Total  Rate')

fig.show()

---
## Portfolio Summary

| Project | Key Finding |
|---------|------------|
| Heatmap | Extractives leads Environment; Financials leads Governance |
| Top Topics | Business Ethics and Data Security material for nearly all industries |
| Deep-Dive | Oil and Gas has 18+ material ESG topics |
| Radar | Tech and Extractives have opposite ESG risk profiles |
| Materiality Rate | Extractives and Infrastructure highest material rate |

**Framework:** SASB Standards | **Tools:** Python, Pandas, Plotly  
*ESG Portfolio - Naveed Baig - 2025*